In [0]:
%sql
-- create cleaned silver schema
USE CATALOG `retail-dwh-project`;

CREATE SCHEMA IF NOT EXISTS clean;


In [0]:

-- CUSTOMERS CLEAN

CREATE OR REPLACE TABLE clean.customers_clean AS
SELECT DISTINCT
    CustomerID,
    INITCAP(TRIM(CustomerName)) AS CustomerName,
    LOWER(TRIM(Email))          AS Email,
    TRIM(City)                  AS City,
    TRIM(Address)               AS Address,
    LastUpdated
FROM bronze.customers_raw;

-- Removes extra spaces and converts proper case
-- Standardizes emails.

In [0]:

-- PRODUCTS CLEAN

CREATE OR REPLACE TABLE clean.products_clean AS
SELECT DISTINCT
    ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category)    AS Category,
    UnitPrice
FROM bronze.products_raw
WHERE UnitPrice > 0;

-- removes invalid products with zero price

In [0]:

-- STORES CLEAN
CREATE OR REPLACE TABLE clean.stores_clean AS
SELECT DISTINCT
    StoreID,
    INITCAP(TRIM(StoreName))          AS StoreName,
    COALESCE(TRIM(Region), 'Unknown') AS Region
FROM bronze.stores_raw;


-- Handles missing regions. If null → Unknown

In [0]:

-- SALES CLEAN

CREATE OR REPLACE TABLE clean.sales_clean AS
SELECT DISTINCT
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity,
    TO_DATE(TxnDate, 'dd-MM-yyyy') AS TxnDate
FROM bronze.sales_raw
WHERE Quantity > 0
  AND CustomerID IN (
      SELECT CustomerID
      FROM clean.customers_clean
  );

  
-- Converts string date into proper date format.


In [0]:
-- CLEAN LAYER VALIDATION
-- COUNT CLEAN TABLES

SELECT 'customers_clean' AS table_name, COUNT(*) AS total_rows
FROM clean.customers_clean

UNION ALL

SELECT 'products_clean', COUNT(*)
FROM clean.products_clean

UNION ALL

SELECT 'stores_clean', COUNT(*)
FROM clean.stores_clean

UNION ALL

SELECT 'sales_clean', COUNT(*)
FROM clean.sales_clean;

VALIDATION CHECKING QUERIES


In [0]:
-- 1. Same CustomerID with updated City / Address

SELECT
    CustomerID,
    COUNT(DISTINCT TRIM(City))    AS city_count,
    COUNT(DISTINCT TRIM(Address)) AS address_count
FROM bronze.customers_raw
GROUP BY CustomerID
HAVING COUNT(DISTINCT TRIM(City)) > 1
    OR COUNT(DISTINCT TRIM(Address)) > 1;

-- detect Multiple customer City/Address changes.


In [0]:


-- 2. City values with leading/trailing spaces

SELECT *
FROM bronze.customers_raw
WHERE City != TRIM(City);



In [0]:

-- 3. Email stored in uppercase
-- Find inconsistent email formatting
SELECT *
FROM bronze.customers_raw
WHERE Email != LOWER(Email);


In [0]:
-- 4. Duplicate customer records

SELECT
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    COUNT(*) AS duplicate_count
FROM bronze.customers_raw
GROUP BY
    CustomerID,
    CustomerName,
    Email,
    City,
    Address
HAVING COUNT(*) > 1;



In [0]:

-- VALIDATION CHECKS : products_src.csv

-- 5. ProductName with extra spaces (Checks extra spaces.)

SELECT *
FROM bronze.products_raw
WHERE ProductName != TRIM(ProductName);



In [0]:

-- 6. UnitPrice = 0

SELECT *
FROM bronze.products_raw
WHERE UnitPrice = 0;



In [0]:

-- VALIDATION CHECKS : stores_src.csv

-- 7. Region missing (Checks null/blank regions.)

SELECT *
FROM bronze.stores_raw
WHERE Region IS NULL
   OR TRIM(Region) = '';


In [0]:


-- 8. StoreName inconsistency (case difference)

SELECT
    LOWER(TRIM(StoreName)) AS normalized_name,
    COUNT(DISTINCT StoreName) AS variations
FROM bronze.stores_raw
GROUP BY LOWER(TRIM(StoreName))
HAVING COUNT(DISTINCT StoreName) > 1;



In [0]:

-- VALIDATION CHECKS : sales_transactions_src.csv

-- 9. CustomerID not present in customer file

SELECT s.*
FROM bronze.sales_raw s
LEFT JOIN bronze.customers_raw c
       ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;

-- Sales without customer reference (Referential integrity issue)

In [0]:

-- 10. Duplicate TransactionID

SELECT
    TransactionID,
    COUNT(*) AS duplicate_count
FROM bronze.sales_raw
GROUP BY TransactionID
HAVING COUNT(*) > 1;

-- Duplicate sales transactions

In [0]:

-- 11. Quantity = 0

SELECT *
FROM bronze.sales_raw
WHERE Quantity = 0;

-- Invalid sales quantity

In [0]:


-- 12. TxnDate format inconsistency

SELECT *
FROM bronze.sales_raw
WHERE TO_DATE(TxnDate, 'dd-MM-yyyy') IS NULL;

-- Checks improper transaction dates (invalid date format)